# RetinaGuard: 01 - Data Exploration & Retinal Preprocessing Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harsha-Vardan/RetinaGuard/blob/main/notebooks/01_Data_Exploration_and_Preprocessing.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-RetinaGuard-blue.svg)](https://github.com/Harsha-Vardan/RetinaGuard)

---

## Notebook Overview
This notebook explores the multi-center clinical diabetic retinopathy datasets used in **RetinaGuard**:
1. **Cohort Analysis**: Demographic and class distributions across **IDRiD** (India, clinical clinic) and **APTOS 2019**.
2. **Resolution & Hardware Disparity**: Analyzing camera variations (4288x2848 Kowa VX-10 vs 2048x1536 portable cameras).
3. **Preprocessing Pipeline**: Circular FOV masking, aspect-ratio preserving padding, and LAB CLAHE contrast enhancement.
4. **Lesion Ground Truths**: Inspecting microaneurysm, hemorrhage, and hard exudate segmentation masks.

### Step 1: Imports and Workspace Setup

In [ ]:
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

# Ensure project root is in Python path
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().joinpath("src").exists() else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessing import preprocess_fundus_image, extract_circular_fov, apply_clahe

print(f"Workspace root: {PROJECT_ROOT}")
sns.set_theme(style="whitegrid")

### Step 2: Load Master Dataset Metadata & Split Manifests
Inspect the patient-stratified cohort distribution across both clinical centers.

In [ ]:
meta_path = PROJECT_ROOT / "data" / "metadata" / "master_metadata.csv"
if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"Total Master Records: {len(df)}")
    display(df.groupby(["dataset_id", "split"]).size().unstack().fillna(0))
else:
    print(f"Metadata file not found at: {meta_path}")

### Step 3: Class Distribution Across ICDR DR Severity Grades
Analyze the prevalence of No DR (0), Mild (1), Moderate (2), Severe (3), and Proliferative DR (4).

In [ ]:
grade_names = ["0: No DR", "1: Mild", "2: Moderate", "3: Severe", "4: Proliferative"]
valid_df = df[df["dr_grade"].notna()].copy()
valid_df["grade_label"] = valid_df["dr_grade"].astype(int).map(lambda g: grade_names[g])

plt.figure(figsize=(10, 5), dpi=120)
ax = sns.countplot(data=valid_df, x="grade_label", hue="dataset_id", palette=["#1B9E77", "#D95F02"])
plt.title("Diabetic Retinopathy Grade Distribution Across Clinical Cohorts", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("ICDR Diabetic Retinopathy Severity Grade", fontsize=11)
plt.ylabel("Number of Images", fontsize=11)
plt.legend(title="Dataset / Hospital Center")

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{int(height)}", (p.get_x() + p.get_width() / 2., height + 10), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

### Step 4: Retinal Preprocessing Comparison (Raw vs. CLAHE + FOV)
Compare raw uncalibrated camera input with RetinaGuard's standardized circular field-of-view (FOV) and LAB CLAHE enhancement.

In [ ]:
# Select a sample clinical image
sample_row = df[(df["dataset_id"] == "IDRiD") & (df["dr_grade"] >= 2)].iloc[0]
sample_path = Path(sample_row["image_path"])

raw_img = cv2.imread(str(sample_path))
raw_rgb = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

# Preprocess using RetinaGuard pipeline
processed_rgb = preprocess_fundus_image(raw_rgb, target_size=(512, 512))

fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=120)
axes[0].imshow(raw_rgb)
axes[0].set_title(f"Raw Camera Image (Resolution: {raw_rgb.shape[1]}x{raw_rgb.shape[0]})\nGrade: {int(sample_row['dr_grade'])} ({sample_row['dataset_id']})", fontsize=11)
axes[0].axis("off")

axes[1].imshow(processed_rgb)
axes[1].set_title("RetinaGuard Standardized FOV + CLAHE (512x512)\nVessel Contrast & Lesion Boundary Enhanced", fontsize=11)
axes[1].axis("off")

plt.suptitle("Retinal Image Normalization Pipeline", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Step 5: Clinical Lesion Segmentation Ground Truths
Inspect pixel-level microaneurysm, hemorrhage, and exudate masks used to supervise RetinaGuard's Spatial Lesion Attention branch.

In [ ]:
masked_samples = df[df["lesion_mask_path"].notna()]
if not masked_samples.empty:
    mask_row = masked_samples.iloc[0]
    img_p = Path(mask_row["image_path"])
    mask_p = Path(mask_row["lesion_mask_path"])

    fundus_img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    lesion_mask = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)

    # Resize for visualization
    fundus_img = cv2.resize(fundus_img, (512, 512))
    lesion_mask = cv2.resize(lesion_mask, (512, 512), interpolation=cv2.INTER_NEAREST)

    # Create lesion overlay in red
    overlay = fundus_img.copy()
    overlay[lesion_mask > 0] = [255, 30, 30]
    blended = cv2.addWeighted(fundus_img, 0.7, overlay, 0.3, 0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=120)
    axes[0].imshow(fundus_img)
    axes[0].set_title("Fundus Photograph (512x512)", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(lesion_mask, cmap="hot")
    axes[1].set_title(f"Ground-Truth Lesion Mask\n(Active Pixels: {np.count_nonzero(lesion_mask)})", fontsize=11)
    axes[1].axis("off")

    axes[2].imshow(blended)
    axes[2].set_title("Clinical Pathological Overlay (Red)", fontsize=11)
    axes[2].axis("off")

    plt.suptitle(f"Pixel-Level Lesion Annotation Ground Truth ({mask_row['dataset_id']})", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No lesion masks found in current metadata.")